# Tool Calling

**Goal:** Run function/tool calling end to end, including the error paths.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks) — the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).


In [ ]:
%pip install -q anthropic

In [ ]:
import os

# In Colab, read the key from Secrets (key icon in the left sidebar).
# Locally, set the ANTHROPIC_API_KEY env var instead.
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except ImportError:
    assert os.environ.get('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY'

import anthropic
client = anthropic.Anthropic()
MODEL = 'claude-sonnet-5'  # good default: capable and cheap enough to iterate on


## The round trip

Tool calling is a protocol, not magic. The model never executes anything — it emits a *request* to call a tool, and your code does the rest:

1. You send `tools=[...]` with the user message.
2. The model replies with `stop_reason == 'tool_use'` and one or more `tool_use` blocks (tool name + parsed arguments + an `id`).
3. **You** execute the function and send back a `tool_result` block referencing that `id`.
4. The model continues — possibly calling more tools — until it answers in plain text (`stop_reason == 'end_turn'`).

The API is stateless: every step resends the whole conversation. Two tools for this notebook — a fake `get_weather` (canned data, so the notebook runs anywhere) and a real `calculate` that safely evaluates arithmetic.


In [ ]:
import ast
import json
import operator

TOOLS = [
    {
        'name': 'get_weather',
        'description': (
            'Get the current weather for a city. Call this whenever the user asks about '
            'weather, temperature, or outdoor conditions in a specific place.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'city': {'type': 'string', 'description': 'City name, e.g. "Tokyo"'},
                'unit': {'type': 'string', 'enum': ['celsius', 'fahrenheit']},
            },
            'required': ['city'],
        },
    },
    {
        'name': 'calculate',
        'description': (
            'Evaluate an arithmetic expression exactly. Call this for any math beyond '
            'trivial mental arithmetic instead of computing in your head. '
            'Supports + - * / ** and parentheses on numbers only.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'expression': {'type': 'string', 'description': 'e.g. "(17.5 * 12) / 3"'},
            },
            'required': ['expression'],
        },
    },
]


# --- implementations ---

FAKE_WEATHER = {
    'tokyo': {'temp_c': 31, 'conditions': 'humid, partly cloudy'},
    'paris': {'temp_c': 24, 'conditions': 'clear'},
    'london': {'temp_c': 18, 'conditions': 'light rain'},
}


def get_weather(city, unit='celsius'):
    data = FAKE_WEATHER.get(city.lower())
    if data is None:
        raise KeyError(f'no weather data for {city!r}')
    temp = data['temp_c'] if unit == 'celsius' else round(data['temp_c'] * 9 / 5 + 32)
    return f"{city}: {temp} degrees {unit}, {data['conditions']}"


_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}


def calculate(expression):
    """Evaluate arithmetic via the AST — never eval() model-provided strings."""
    def ev(node):
        if isinstance(node, ast.Expression):
            return ev(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError(f'unsupported expression: {expression!r}')
    return ev(ast.parse(expression, mode='eval'))


HANDLERS = {'get_weather': get_weather, 'calculate': calculate}


## Step 1: see the raw tool request

Before building the loop, look at what the model actually sends back. `block.input` arrives as an already-parsed dict (the SDK handles the JSON), and `block.id` is the handle you'll echo back with the result.


In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=500,
    tools=TOOLS,
    messages=[{'role': 'user', 'content': 'What is the weather in Tokyo right now?'}],
)

print('stop_reason:', response.stop_reason)
for block in response.content:
    if block.type == 'text':
        print('text:', block.text)
    elif block.type == 'tool_use':
        print(f'tool_use: {block.name}({json.dumps(block.input)})  id={block.id}')


Run it and note `stop_reason` is `tool_use` — the model has paused mid-turn, waiting for you. There may also be a short `text` block before the tool call (the model narrating its plan); that's normal.

## Step 2: the full loop

Now close the circuit. Three rules most tutorials get right, and two they skip:

- Append the assistant's **entire** `response.content` (not just the text) before the results — the `tool_use` blocks must stay in history.
- Return **all** tool results in a **single** user message, one `tool_result` per `tool_use`, matched by `tool_use_id`.
- Cap the loop. A model that keeps calling tools forever should hit your ceiling, not your credit card.

Skipped by tutorials: exceptions (wrap the handler; return `is_error: True` instead of crashing) and parallel calls (one response can contain *several* `tool_use` blocks — handle all of them, not just the first).


In [ ]:
def execute_tool(block):
    """Run one tool_use block; always returns a tool_result (error or not)."""
    try:
        handler = HANDLERS[block.name]
        result = handler(**block.input)
        return {'type': 'tool_result', 'tool_use_id': block.id, 'content': str(result)}
    except Exception as e:
        # Never let a tool exception kill the loop — report it to the model instead.
        return {'type': 'tool_result', 'tool_use_id': block.id,
                'content': f'{type(e).__name__}: {e}', 'is_error': True}


def run_agent(user_message, max_turns=5, verbose=True):
    # At most max_turns API calls per invocation.
    messages = [{'role': 'user', 'content': user_message}]
    for turn in range(max_turns):
        response = client.messages.create(
            model=MODEL, max_tokens=600, tools=TOOLS, messages=messages,
        )
        if response.stop_reason != 'tool_use':
            return next((b.text for b in response.content if b.type == 'text'), '')

        messages.append({'role': 'assistant', 'content': response.content})
        results = []
        for block in response.content:
            if block.type == 'tool_use':
                result = execute_tool(block)
                if verbose:
                    flag = ' [ERROR]' if result.get('is_error') else ''
                    print(f"  turn {turn}: {block.name}({json.dumps(block.input)}) "
                          f"-> {result['content']}{flag}")
                results.append(result)
        messages.append({'role': 'user', 'content': results})  # ALL results, ONE message
    raise RuntimeError(f'agent did not finish within {max_turns} turns')


print(run_agent('What is the weather in Tokyo, and what is 17.5% of 2340?'))


## Error path 1: the tool raises

Ask about a city our fake weather source doesn't know. `get_weather` raises `KeyError`, `execute_tool` converts it to a `tool_result` with `is_error: True`, and the model gets to decide what to do — apologize, ask for clarification, or try something else. Run it and note the model produces a graceful answer instead of the loop crashing.


In [ ]:
print(run_agent('What is the weather in Reykjavik?'))


The quality of the error *message* matters as much as the flag. `KeyError: 'Reykjavik'` is decodable; a bare stack trace or `Error 500` gives the model nothing to recover with. Write tool errors the way you'd write them for a junior engineer reading logs: what failed, why, and what a valid retry would look like.

## Error path 2: invalid arguments

The model can also call *your* tool wrong — an expression your calculator doesn't support, a missing field, a wrong type. Two layers of defense:

- **Your handler validates.** `calculate` rejects anything that isn't pure arithmetic (run the cell below: the model relays the tool's complaint or retries with a fixed expression).
- **The API can validate for you.** Add `'strict': True` to a tool definition (with `additionalProperties: False` and a `required` list) and the API guarantees `block.input` matches the schema exactly — wrong-shaped calls never reach your code. Schema validation still won't catch *semantically* bad values, so keep the handler checks too.


In [ ]:
# 'x' is not defined, so the AST evaluator rejects the expression.
# Run this and watch the error round trip: is_error -> model recovers or explains.
print(run_agent('Use the calculate tool to evaluate "x + 2" where x is 5.'))


## Parallel tool calls

By default the model may request several tools in one response — one assistant message containing multiple `tool_use` blocks. Our loop already handles this because it iterates all blocks and returns all results in a single user message. That last part is load-bearing: splitting results across multiple user messages quietly teaches the model to stop parallelizing.


In [ ]:
# A question that invites three tool calls at once. Watch how many happen per turn.
print(run_agent(
    'Compare the current weather in Tokyo, Paris, and London. '
    'Then tell me the average of their temperatures in celsius (use calculate).'
))


Run it and note the turn numbers in the trace: the three `get_weather` calls typically land in the *same* turn (parallel), then `calculate` follows in the next one because it depends on their results. The model figured out that dependency ordering on its own.

## Tool descriptions are prompts

The single highest-leverage line in a tool definition is the `description`. The model decides *whether* and *how* to call your tool almost entirely from it. Vague descriptions produce wrong calls:

- `"Gets weather"` — under-triggers (model answers from memory) and invites wrong arguments (country instead of city).
- `"Get the current weather for a city. Call this whenever the user asks about weather... "` — states *when* to call it, not just what it does. Trigger conditions in the description measurably improve call rates on current models, which are conservative about reaching for tools.

Same discipline as API docs: describe each parameter, use `enum` for closed sets, mark only truly required fields as required. If the model keeps misusing a tool, fix the description before you fix the prompt.


## Exercises

1. Add a third tool `convert_currency(amount, from_ccy, to_ccy)` backed by a hardcoded rate table, then ask a question that needs weather + currency + math in one request. Check the trace for parallelism.
2. Make `get_weather` fail randomly 50% of the time with a `TimeoutError`, and extend `execute_tool` to retry once before returning `is_error`. Compare transcripts with and without the retry.
3. Add `'strict': True` (plus `additionalProperties: False` and full `required` lists) to both tool definitions, then try to provoke a malformed call. Verify bad shapes no longer reach your handlers.
4. Rewrite the `calculate` description to be maximally vague ("does math stuff") and re-run the percentage question 3 times. Count how often the model computes in its head instead of calling the tool.
